In [0]:
-- =========================================================
-- CFPB Complaint Risk Control Tower
-- Demo workflow state seeding script (historical timestamp version)
-- =========================================================

-- =========================================================
-- STEP 0: clear app-layer state
-- =========================================================

TRUNCATE TABLE cfpb_risk.app.issue_events;

TRUNCATE TABLE cfpb_risk.app.issues;



-- =========================================================
-- STEP 1: repopulate app.issues from gold.risk_alerts
-- with seeded historical created_ts / updated_ts
-- =========================================================

INSERT INTO cfpb_risk.app.issues (
  issue_id,
  alert_id,
  institution_display_name,
  rssd_id,
  complaint_month,
  product,
  issue,
  risk_score,
  alert_level,
  alert_reason,
  current_status,
  priority,
  issue_owner,
  due_date,
  latest_note,
  created_ts,
  updated_ts
)
WITH base_alerts AS (
  SELECT
    sha2(concat_ws('|', alert_id, institution_display_name, product, issue, cast(complaint_month as string)), 256) AS issue_id,
    alert_id,
    institution_display_name,
    rssd_id,
    complaint_month,
    product,
    issue,
    risk_score,
    alert_level,
    alert_reason
  FROM cfpb_risk.gold.risk_alerts
),
seeded_dates AS (
  SELECT
    *,
    pmod(abs(hash(concat(issue_id, '|created_day'))), 90) AS created_day_offset,
    pmod(abs(hash(concat(issue_id, '|created_hour'))), 9) AS created_hour_offset,
    pmod(abs(hash(concat(issue_id, '|created_minute'))), 60) AS created_minute_offset
  FROM base_alerts
)
SELECT
  issue_id,
  alert_id,
  institution_display_name,
  rssd_id,
  complaint_month,
  product,
  issue,
  risk_score,
  alert_level,
  alert_reason,
  'New' AS current_status,
  CASE
    WHEN alert_level IN ('Urgent', 'High') THEN 'High'
    WHEN alert_level = 'Medium' THEN 'Medium'
    ELSE 'Low'
  END AS priority,
  NULL AS issue_owner,
  NULL AS due_date,
  NULL AS latest_note,
  (
    CAST(date_sub(current_date(), created_day_offset) AS timestamp) + INTERVAL 8 HOURS + MAKE_INTERVAL(0, 0, 0, 0, created_hour_offset, created_minute_offset, 0)
  ) AS created_ts,
  (
    CAST(date_sub(current_date(), created_day_offset) AS timestamp)
    + INTERVAL 8 HOURS
    + MAKE_INTERVAL(0, 0, 0, 0, created_hour_offset, created_minute_offset, 0)
  ) AS updated_ts
FROM seeded_dates;



-- =========================================================
-- STEP 2: seed current-state workflow values deterministically
-- using created_ts-aware due dates and realistic updated_ts
-- =========================================================

MERGE INTO cfpb_risk.app.issues AS tgt
USING (
  WITH seeded_base AS (
    SELECT
      issue_id,
      alert_level,
      created_ts,
      pmod(abs(hash(issue_id)), 100) AS status_bucket,
      pmod(abs(hash(concat(issue_id, '|owner'))), 100) AS owner_bucket,
      pmod(abs(hash(concat(issue_id, '|due_date'))), 100) AS due_bucket,
      pmod(abs(hash(concat(issue_id, '|note'))), 100) AS note_bucket
    FROM cfpb_risk.app.issues
  ),
  seeded_values AS (
    SELECT
      issue_id,
      created_ts,

      CASE
        WHEN status_bucket < 45 THEN 'New'
        WHEN status_bucket < 75 THEN 'In Review'
        WHEN status_bucket < 90 THEN 'Escalated'
        ELSE 'Closed'
      END AS seeded_status,

      CASE
        WHEN alert_level IN ('Urgent', 'High') THEN 'High'
        WHEN alert_level = 'Medium' THEN 'Medium'
        ELSE 'Low'
      END AS seeded_priority,

      CASE
        WHEN owner_bucket < 20 THEN NULL
        WHEN owner_bucket < 45 THEN 'alex@bank.demo'
        WHEN owner_bucket < 70 THEN 'jordan@bank.demo'
        WHEN owner_bucket < 85 THEN 'casey@bank.demo'
        ELSE 'taylor@bank.demo'
      END AS seeded_owner,

      CASE
        WHEN note_bucket < 30 THEN 'Initial triage completed; issue moved into review.'
        WHEN note_bucket < 50 THEN 'Pattern appears concentrated in complaint volume this month.'
        WHEN note_bucket < 65 THEN 'Ownership assigned for follow-up and investigation.'
        WHEN note_bucket < 80 THEN 'Escalated for additional compliance review.'
        ELSE NULL
      END AS seeded_note,

      due_bucket
    FROM seeded_base
  )
  SELECT
    issue_id,
    seeded_status,
    seeded_priority,
    seeded_owner,
    seeded_note,

    CASE
      WHEN due_bucket < 20 THEN date_add(to_date(created_ts), 3)
      WHEN due_bucket < 40 THEN date_add(to_date(created_ts), 7)
      WHEN due_bucket < 65 THEN date_add(to_date(created_ts), 14)
      WHEN due_bucket < 85 THEN date_add(to_date(created_ts), 21)
      ELSE date_add(to_date(created_ts), 30)
    END AS seeded_due_date,

    CASE
      WHEN seeded_status = 'New' THEN created_ts
      WHEN seeded_status = 'In Review' THEN created_ts + INTERVAL 2 DAYS
      WHEN seeded_status = 'Escalated' THEN created_ts + INTERVAL 4 DAYS
      WHEN seeded_status = 'Closed' THEN created_ts + INTERVAL 6 DAYS
      ELSE created_ts
    END AS seeded_updated_ts

  FROM seeded_values
) AS src
ON tgt.issue_id = src.issue_id
WHEN MATCHED THEN UPDATE SET
  tgt.current_status = src.seeded_status,
  tgt.priority = src.seeded_priority,
  tgt.issue_owner = src.seeded_owner,
  tgt.due_date = src.seeded_due_date,
  tgt.latest_note = src.seeded_note,
  tgt.updated_ts = src.seeded_updated_ts;



-- =========================================================
-- STEP 3: rebuild audit/event timeline from seeded state
-- =========================================================

TRUNCATE TABLE cfpb_risk.app.issue_events;



-- ---------------------------------------------------------
-- 3A. CREATED events
-- ---------------------------------------------------------

INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  institution_display_name,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'CREATED', cast(i.created_ts as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts AS event_ts,
  'CREATED' AS event_type,
  'system@demo' AS event_user,
  NULL AS old_status,
  'New' AS new_status,
  NULL AS old_priority,
  i.priority AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  i.due_date AS new_due_date,
  NULL AS note_text,
  'Issue created from scored alert' AS event_comment
FROM cfpb_risk.app.issues i;



-- ---------------------------------------------------------
-- 3B. OWNER_ASSIGNED events
-- ---------------------------------------------------------

INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  institution_display_name,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'OWNER_ASSIGNED', cast(i.created_ts + INTERVAL 1 DAY as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts + INTERVAL 1 DAY AS event_ts,
  'OWNER_ASSIGNED' AS event_type,
  'manager@bank.demo' AS event_user,
  NULL AS old_status,
  NULL AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  i.issue_owner AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  NULL AS note_text,
  'Issue assigned to owner for review' AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.issue_owner IS NOT NULL;



-- ---------------------------------------------------------
-- 3C. STATUS_CHANGED -> In Review
-- ---------------------------------------------------------

INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  institution_display_name,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'STATUS_CHANGED', 'IN_REVIEW', cast(i.created_ts + INTERVAL 2 DAY as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts + INTERVAL 2 DAY AS event_ts,
  'STATUS_CHANGED' AS event_type,
  coalesce(i.issue_owner, 'manager@bank.demo') AS event_user,
  'New' AS old_status,
  'In Review' AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  NULL AS note_text,
  'Issue moved into active review' AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.current_status IN ('In Review', 'Escalated', 'Closed');



-- ---------------------------------------------------------
-- 3D. NOTE_ADDED events
-- ---------------------------------------------------------

INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  institution_display_name,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'NOTE_ADDED', cast(i.created_ts + INTERVAL 3 DAY as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts + INTERVAL 3 DAY AS event_ts,
  'NOTE_ADDED' AS event_type,
  coalesce(i.issue_owner, 'analyst@bank.demo') AS event_user,
  NULL AS old_status,
  NULL AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  i.latest_note AS note_text,
  'Issue note added' AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.latest_note IS NOT NULL;



-- ---------------------------------------------------------
-- 3E. STATUS_CHANGED -> Escalated
-- ---------------------------------------------------------

INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  institution_display_name,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'STATUS_CHANGED', 'ESCALATED', cast(i.created_ts + INTERVAL 4 DAY as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts + INTERVAL 4 DAY AS event_ts,
  'STATUS_CHANGED' AS event_type,
  coalesce(i.issue_owner, 'manager@bank.demo') AS event_user,
  'In Review' AS old_status,
  'Escalated' AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  NULL AS note_text,
  'Issue escalated for additional attention' AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.current_status IN ('Escalated', 'Closed');



-- ---------------------------------------------------------
-- 3F. STATUS_CHANGED -> Closed
-- ---------------------------------------------------------

INSERT INTO cfpb_risk.app.issue_events (
  event_id,
  issue_id,
  alert_id,
  institution_display_name,
  event_ts,
  event_type,
  event_user,
  old_status,
  new_status,
  old_priority,
  new_priority,
  old_owner,
  new_owner,
  old_due_date,
  new_due_date,
  note_text,
  event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'STATUS_CHANGED', 'CLOSED', cast(i.created_ts + INTERVAL 6 DAY as string)), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts + INTERVAL 6 DAY AS event_ts,
  'STATUS_CHANGED' AS event_type,
  coalesce(i.issue_owner, 'manager@bank.demo') AS event_user,
  'In Review' AS old_status,
  'Closed' AS new_status,
  NULL AS old_priority,
  NULL AS new_priority,
  NULL AS old_owner,
  NULL AS new_owner,
  NULL AS old_due_date,
  NULL AS new_due_date,
  NULL AS note_text,
  'Issue closed' AS event_comment
FROM cfpb_risk.app.issues i
WHERE i.current_status = 'Closed';